# EDA Dataset EmoEvent (español)

Este notebook analiza el dataset **EmoEvent** (Plaza et al., 2020) en su partición española.   Se carga directamente desde el CSV oficial del grupo SINAI (`emoevent_es.csv`), sin dependencia de la API de HuggingFace ni del subconjunto en inglés.

## 0. Imports y configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import textwrap

def save_custom_plot(filename, dpi=300):
    """
    Guarda la figura activa de matplotlib en el directorio de resultados.
    Asegura alta resolución y recorta los márgenes blancos innecesarios.
    """
    output_dir = Path('../docs/figures')
    
    # Aseguramos que la extensión .png esté presente
    if not filename.endswith('.png'):
        filename += '.png'
        
    plt.savefig(output_dir / filename, bbox_inches='tight', dpi=dpi)
    print(f"Gráfica guardada: {filename}")


# Estilo global
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams.update({'figure.dpi': 130, 'figure.facecolor': 'white'})

# Constantes y configuración de reproducibilidad
SEED = 42
DATA_DIR = Path('../data/raw/emoevent')
CSV_PATH = DATA_DIR / 'emoevent_es.csv'

# Configuración visual de las clases
EMOTION_ORDER = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'others']
PALETTE = sns.color_palette('tab10', n_colors=len(EMOTION_ORDER))
EMO_COLOR = dict(zip(EMOTION_ORDER, PALETTE))

## 1. Carga del dataset (español)

In [ ]:
# Carga directa del CSV español
df_raw = pd.read_csv(CSV_PATH, encoding='utf-8', sep="\t")

print(f'Shape inicial: {df_raw.shape}')
print(f'Columnas originales: {df_raw.columns.tolist()}')
df_raw.head(3)

In [ ]:
# Normalizar nombres de columnas a texto / emotion / offensive
col_map = {'tweet': 'text', 'emotion': 'emotion', 'offensive': 'offensive'}
df = df_raw.rename(columns=col_map).copy()

# Limpieza básica: Eliminar duplicados y nulos críticos antes de procesar
df = df.drop_duplicates(subset=['text'])
df = df.dropna(subset=['text', 'emotion'])

# Convertir etiquetas numéricas de emoción
if pd.api.types.is_integer_dtype(df['emotion']):
    emo_int_map = {0:'anger', 1:'disgust', 2:'fear', 3:'joy', 4:'sadness', 5:'surprise', 6:'others'}
    df['emotion'] = df['emotion'].map(emo_int_map)

# Convertir ofensividad numérica a string legible
if 'offensive' in df.columns and pd.api.types.is_integer_dtype(df['offensive']):
    df['offensive'] = df['offensive'].map({0: 'no', 1: 'yes'})

# Features auxiliares de longitud
df['text_len'] = df['text'].str.split().str.len()
df['char_len'] = df['text'].str.len()

# Mostrar informacióon
print(f'Total registros en español : {len(df):,}')
print(f'Emociones presentes : {sorted(df["emotion"].dropna().unique())}')
print(f'Valores de ofensividad : {df["offensive"].unique()}')
df[['text', 'emotion', 'offensive', 'text_len']].head()

## 2. Distribución de clases emocionales

In [ ]:
# Obtener los conteo y porcentajes de aparición de cada emoción
emo_counts = df['emotion'].value_counts().reindex(EMOTION_ORDER, fill_value=0)
emo_pct = (emo_counts / emo_counts.sum() * 100).round(1)

# Gráfica
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    emo_counts.index,
    emo_counts.values,
    color=[EMO_COLOR[e] for e in emo_counts.index],
    edgecolor='white', linewidth=0.8
)
labels = [f'{p}%' for p in emo_pct.values]
ax.bar_label(bars, labels=labels, padding=3, fontsize=10, fontweight='bold')
ax.set_title('Distribución de emociones — EmoEvent (español)', fontsize=13, pad=12)
ax.set_xlabel('Emoción', labelpad=8)
ax.set_ylabel('N.º de muestras', labelpad=8)
sns.despine(ax=ax)
plt.tight_layout()

# Guardar la gráfica en /docs/figures
save_custom_plot('emo_distribution')
plt.show()

# Distribución absoluta de cada emoción
print('\nConteos absolutos:')
print(pd.DataFrame({'count': emo_counts, 'pct': emo_pct}).to_string())

Se puede observar un fuerte desequilibrio de clases en el dataset. La categoría mayoritaria es others (49.1%), abarcando prácticamente la mitad del corpus, la cual agrupa mensajes neutrales o que no encajan en las emociones básicas de Ekman.

Entre las emociones básicas, destaca la prevalencia de joy (21.6%), seguida de sadness (12.0%) y anger (10.2%). Por el contrario, las clases minoritarias: disgust (1.9%) y fear (1.1%) están severamente subrepresentadas. Las más importantes para un chatbot centrado en la gente que sufre ciberacoso serían sadness y fear, por lo que habrá que mitigar este desequilibrio

## 3. Ratio de desequilibrio de clases

In [ ]:
# Extracción de frecuencias absolutas
max_class_frequency = emo_counts.max()
min_class_frequency = emo_counts[emo_counts > 0].min()

# Cálculo del Imbalance Ratio Global (Clase mayoritaria vs Clase minoritaria)
global_imbalance_ratio = max_class_frequency / min_class_frequency

# Generación de tabla con el IR específico por cada emoción. El IR de cada clase se calcula respecto a la clase mayoritaria
df_imbalance_metrics = pd.DataFrame({
    'sample_count': emo_counts,
    'imbalance_ratio': (max_class_frequency / emo_counts.replace(0, np.nan)).round(2)
}).sort_values('imbalance_ratio', ascending=False)

# Impresión de métricas
print(f"Ratio Global (Max/Min): {global_imbalance_ratio:.2f}\n")
print("Detalle de Imbalance Ratio por clase:")
print(df_imbalance_metrics.to_string())

Al cuantificar este desequilibrio mediante el Imbalance Ratio (IR), observamos un ratio global severo de aproximadamente 43x entre la clase mayoritaria (others) y la minoritaria (fear).

## 4. Longitud de texto por emoción (boxplot)

In [ ]:
# Asegurar el orden visual consistente
order = [e for e in EMOTION_ORDER if e in df['emotion'].unique()]

# Configuración de la figura con dos subgráficos (palabras y caracteres)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Iteración sobre las métricas para generar los boxplots dinámicamente
for ax, metric, label, unit in zip(
    axes,
    ['text_len', 'char_len'],
    ['Longitud en tokens (palabras)', 'Longitud en caracteres'],
    ['palabras', 'caracteres']
):
    sns.boxplot(
        data=df[df['emotion'].isin(order)],
        x='emotion', 
        y=metric,
        order=order,
        hue='emotion',
        palette=EMO_COLOR,
        legend=False,
        flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.4},
        linewidth=0.9,
        ax=ax
    )
    # Formato individual de cada subgráfico
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Emoción', labelpad=6)
    ax.set_ylabel(f'N.º de {unit}', labelpad=6)
    ax.tick_params(axis='x', rotation=30)
    sns.despine(ax=ax)

# Ajustes finales y guardado de la figura
fig.suptitle('Longitud de texto por emoción — EmoEvent (español)', fontsize=13, y=1.02)
plt.tight_layout()

save_custom_plot('text_length_per_emotion')
plt.show()

# Estadísticos descriptivos
len_stats = df.groupby('emotion')['text_len'].describe().round(1)
print('\nEstadísticos de longitud (palabras):')
print(len_stats.to_string())

p95 = df['text_len'].quantile(0.95)
p99 = df['text_len'].quantile(0.99)
print(f'\nPercentil 95 global: {p95:.0f} palabras')
print(f'Percentil 99 global: {p99:.0f} palabras')
print(f'Max global : {df["text_len"].max()} palabras')

El corpus está compuesto por textos muy breves, lo cual es coherente con la naturaleza de la plataforma de origen (Twitter). La media global ronda las 20-27 palabras, siendo las emociones de ira (anger) y tristeza (sadness) las que presentan textos ligeramente más extensos y con mayor dispersión.

A nivel técnico, los estadísticos de percentiles son altamente favorables para la arquitectura del sistema. El 99% de los mensajes tiene 49 palabras o menos, y el máximo absoluto es de 59 palabras. Esta brevedad de mensajes nos permite tener un espacio holgado en el prompt para recuperar los chunks terapeúticos definidos manualmente y así garantizamos que la totalidad de la ventana de contexto del SLM quede libre para inyectar íntegramente la información terapéutica recuperada y las instrucciones del sistema, sin riesgo de truncamiento.


## 5. Co-ocurrencia emoción × ofensividad (heatmap)

In [ ]:
# Asegurar el orden visual consistente
order_emo = [e for e in EMOTION_ORDER if e in df['emotion'].unique()]

# Generación de matriz de contingencia absoluta
crosstab_absolute = pd.crosstab(
    df['emotion'], df['offensive']
).reindex(index=order_emo, fill_value=0)

# Generación de matriz de contingencia relativa (normalización por fila). Se calcula la proporción de ofensividad intra-clase emocional
crosstab_percentage = crosstab_absolute.div(
    crosstab_absolute.sum(axis=1), axis=0
).mul(100).round(1)

# Configuración del lienzo visual (Heatmaps gemelos)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap 1: Conteos Absolutos (Escala cálida)
sns.heatmap(
    crosstab_absolute, annot=True, fmt='d', cmap='YlOrRd',
    linewidths=0.5, linecolor='white',
    ax=axes[0], cbar_kws={'label': 'N.º de Muestras'}
)
axes[0].set_title('Conteos Absolutos', fontsize=11)
axes[0].set_xlabel('Ofensividad', labelpad=6)
axes[0].set_ylabel('Emoción', labelpad=6)

# Heatmap 2: Porcentajes Relativos (Escala fría)
sns.heatmap(
    crosstab_percentage, annot=True, fmt='.1f', cmap='Blues',
    linewidths=0.5, linecolor='white',
    ax=axes[1], cbar_kws={'label': '% dentro de la emoción'},
    vmin=0, vmax=100
)
axes[1].set_title('Distribución Relativa (% por emoción)', fontsize=11)
axes[1].set_xlabel('Ofensividad', labelpad=6)
axes[1].set_ylabel('')

# Ajustes finales y exportación
fig.suptitle('Co-ocurrencia Emoción x Ofensividad - EmoEvent (español)',
                fontsize=13, y=1.02)
plt.tight_layout()

save_custom_plot('emotion_offensive_heatmap')
plt.show()

# 6. Extracción de métricas puras
print('Métricas de Contingencia: % Relativo')
print(crosstab_percentage.to_string())

# Extracción del valor máximo para el log
off_col = 'yes' if 'yes' in crosstab_percentage.columns else crosstab_percentage.columns[-1]
most_offensive_emotion = crosstab_percentage[off_col].idxmax()
max_offensive_rate = crosstab_percentage.loc[most_offensive_emotion, off_col]

print(f'Métrica max_offensive_rate : {max_offensive_rate}%')
print(f'Métrica max_offensive_class : {most_offensive_emotion}')

Como era previsible en un contexto de hostilidad digital, las emociones asociadas a la agresividad (disgust y anger) presentan las mayores tasas de lenguaje ofensivo.
Por el contrario, emociones como la tristeza (sadness) o el miedo (fear), aunque minoritarias, presentan perfiles distintos.

Esta distinción empírica valida la necesidad de un enfoque dual (clasificador emocional + sistema RAG). Un mensaje puede no contener lenguaje estrictamente "ofensivo" o baneable por los filtros tradicionales, pero el clasificador puede etiquetarlo como fear o sadness. Esta etiqueta actuará como metadato clave en nuestra base de datos vectorial, permitiendo al sistema RAG recuperar el chunk con la técnica terapéutica de contención exacta diseñada para ese estado de vulnerabilidad emocional, ofreciendo una respuesta mucho más adaptativa que un simple sistema de bloqueo de insultos.

## 6. Ejemplos de tweets por emoción

In [ ]:
print('Muestreo Cualitativo de Textos por Emoción')

for emotion_label in EMOTION_ORDER:
    emotion_subset = df[df['emotion'] == emotion_label]
    
    if emotion_subset.empty:
        continue
        
    # Extracción de una muestra reproducible
    sample_text = emotion_subset['text'].sample(n=1, random_state=SEED).values[0]
    
    # Formateo del texto para evitar cortes abruptos en medio de una palabra
    formatted_text = textwrap.shorten(str(sample_text), width=180, placeholder=" [...]")
    
    print(f'\n[{emotion_label.upper()}]')
    print(f'  {formatted_text}')

A través de esta inspección se evidencia el ruido inherente a los canales de redes sociales: uso intensivo de HASHTAGS, URL, .... Para que el chatbot actúe como una herramienta efectiva de apoyo frente al ciberacoso, el clasificador subyacente debe ser robusto frente a este ruido